In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)
sparql.addCustomHttpHeader(
    "User-Agent",
    "MyResearchProject/1.0 (your_real_email@example.com)"
)
query = """
SELECT ?company ?companyLabel ?companyId ?exchangeLabel
       (SAMPLE(?ticker0) AS ?ticker)
       (SAMPLE(?industryLabel0) AS ?industry)
       (SAMPLE(?marketCap0) AS ?marketCap)
       (SAMPLE(?marketCapDate0) AS ?marketCapDate)
       (SAMPLE(?revenue0) AS ?revenue)
       (SAMPLE(?revenueDate0) AS ?revenueDate)
WHERE {
  VALUES ?targetExchange { wd:Q13677 wd:Q82059 }   # NYSE, Nasdaq

  ?company p:P414 ?listingStatement .
  ?listingStatement ps:P414 ?targetExchange .

  OPTIONAL { ?listingStatement pq:P249 ?ticker0 . }

  OPTIONAL {
    ?company wdt:P452 ?industry0 .
    ?industry0 rdfs:label ?industryLabel0 .
    FILTER(LANG(?industryLabel0) = "en")
  }

  OPTIONAL {
    {
      SELECT ?company ?marketCap0 ?marketCapDate0 WHERE {
        {
          SELECT ?company (MAX(?capDate) AS ?marketCapDate0) WHERE {
            ?company p:P2226 ?capStmt .
            ?capStmt pq:P585 ?capDate .
          }
          GROUP BY ?company
        }

        ?company p:P2226 ?capStmt .
        ?capStmt pq:P585 ?marketCapDate0 ;
                 psv:P2226 ?capValue .

        ?capValue wikibase:quantityAmount ?marketCap0 .
      }
    }
  }
  OPTIONAL {
  {
    SELECT ?company ?revenue0 ?revenueDate0 WHERE {
      {
        SELECT ?company (MAX(?revDate) AS ?revenueDate0) WHERE {
          ?company p:P2139 ?revStmt .
          ?revStmt pq:P585 ?revDate .
        }
        GROUP BY ?company
      }

      ?company p:P2139 ?revStmt .
      ?revStmt pq:P585 ?revenueDate0 ;
               psv:P2139 ?revValue .

      ?revValue wikibase:quantityAmount ?revenue0 .
    }
  }
}

  BIND(REPLACE(STR(?company), "http://www.wikidata.org/entity/", "") AS ?companyId)

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en".
    ?company rdfs:label ?companyLabel .
    ?targetExchange rdfs:label ?exchangeLabel .
  }
}
GROUP BY ?company ?companyLabel ?companyId ?exchangeLabel
ORDER BY ?companyLabel
LIMIT 10000
"""

sparql.setQuery(query)
results = sparql.query().convert()

rows = []
for row in results["results"]["bindings"]:
    rows.append({
        "company": row.get("companyLabel", {}).get("value"),
        "company_id": row.get("companyId", {}).get("value"),
        "exchange": row.get("exchangeLabel", {}).get("value"),
        "ticker": row.get("ticker", {}).get("value"),
        "industry": row.get("industry", {}).get("value"),
        "market_cap": row.get("marketCap", {}).get("value"),
        "market_cap_date": row.get("marketCapDate", {}).get("value"),
        "company_url": row.get("company", {}).get("value"),
        "revenue": row.get("revenue", {}).get("value"),
        "revenue_date": row.get("revenueDate", {}).get("value"),
    })

df = pd.DataFrame(rows)

# optional cleanup
df["market_cap"] = pd.to_numeric(df["market_cap"], errors="coerce")
df["market_cap_date"] = pd.to_datetime(df["market_cap_date"], errors="coerce")
df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")
df["revenue_date"] = pd.to_datetime(df["revenue_date"], errors="coerce")

df = df.dropna(subset=["revenue"])
df = df.sort_values("revenue", ascending=False).reset_index(drop=True)

df.to_csv("nyse_nasdaq_companies_with_revenue.csv", index=False, encoding="utf-8")
print(df.head(20))
print("Total rows:", len(df))

                                 company company_id                 exchange  \
0                                  Honda      Q9584  New York Stock Exchange   
1                                 Nissan     Q20165                   Nasdaq   
2                                 Itochu    Q717093                   Nasdaq   
3                             Sony Group     Q41187  New York Stock Exchange   
4                             NTT DoCoMo    Q853958  New York Stock Exchange   
5                             Canon Inc.     Q62621  New York Stock Exchange   
6              Micro Focus International   Q1931458  New York Stock Exchange   
7                                Sinopec    Q831445  New York Stock Exchange   
8             PetroChina Company Limited    Q503182  New York Stock Exchange   
9                                   TSMC    Q713418  New York Stock Exchange   
10                          Nomura Group    Q658089  New York Stock Exchange   
11                         América Móvil

In [ ]:
import pandas as pd
import yfinance as yf

target_date = pd.Timestamp("2018-01-01")

def market_cap_on_or_before_date(ticker):
    if pd.isna(ticker) or str(ticker).strip() == "":
        return None

    try:
        t = yf.Ticker(str(ticker).strip())

        price_hist = t.history(
            start=(target_date - pd.Timedelta(days=10)).strftime("%Y-%m-%d"),
            end=(target_date + pd.Timedelta(days=2)).strftime("%Y-%m-%d"),
            auto_adjust=False
        )
        if price_hist.empty:
            return None

        price_hist.index = pd.to_datetime(price_hist.index).tz_localize(None)
        price_hist = price_hist[price_hist.index <= target_date]
        if price_hist.empty:
            return None

        close_price = float(price_hist.iloc[-1]["Close"])

        shares = t.get_shares_full(
            start=(target_date - pd.Timedelta(days=370)).strftime("%Y-%m-%d"),
            end=(target_date + pd.Timedelta(days=2)).strftime("%Y-%m-%d"),
        )
        if shares is None or len(shares) == 0:
            return None

        shares.index = pd.to_datetime(shares.index).tz_localize(None)
        shares = shares[shares.index <= target_date]
        if shares.empty:
            return None

        shares_outstanding = float(shares.iloc[-1])

        return close_price * shares_outstanding

    except Exception:
        return None

df["market_cap_2026_01_01"] = df["ticker"].apply(market_cap_on_or_before_date)

$QGEN: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: GLAE"}}}
$GLAE: possibly delisted; no timezone found
$JEF: possibly delisted; no timezone found
$SI: possibly delisted; no timezone found
$SIAL: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TEF"}}}
$TEF: possibly delisted; no timezone found
$KLAC: possibly delisted; no timezone found
$TEL: possibly delisted; no timezone found
$CMG: possibly delisted; no timezone found
$APH: possibly delisted; no timezone found
$LPL: possibly delisted; no timezone found
$BBRY: possibly delisted; no timezone found
$BB: possibly delisted; no timezone found
$MGA: possibly delisted; no timezone found
$CLS: possibly delisted; no timezone found
$AUO: possibly delisted; no timezone found
$DOX: possibly delisted; no timezone found
$DOX: possibl

In [ ]:
df.to_csv("nyse_nasdaq_companies_cap.csv", index=False, encoding="utf-8")


In [41]:
df_with_market_cap = df[df["market_cap_2026_01_01"].notna()].copy()

print("Number of companies with market cap:", len(df_with_market_cap))

df_with_market_cap = df_with_market_cap.sort_values(
    "market_cap_2026_01_01",
    ascending=False
).reset_index(drop=True)

print(df_with_market_cap.head(20))

Number of companies with market cap: 1056
                   company company_id ticker                  industry  \
0                  Clarcor  Q16958713    CLC               gas turbine   
1             GE Aerospace   Q1485061     GE        aerospace industry   
2         General Electric     Q54173     GE    mechanical engineering   
3            Alibaba Group   Q1359568   BABA                e-commerce   
4           JPMorgan Chase    Q192314    JPM        financial services   
5        Johnson & Johnson    Q333718    JNJ   pharmaceutical industry   
6               ExxonMobil    Q156238    XOM        petroleum industry   
7          Bank of America    Q487907    BAC        financial services   
8              Wells Fargo    Q744149    WFC        financial services   
9                     Visa    Q328840      V        financial services   
10     Chevron Corporation    Q319642    CVX        petroleum industry   
11        Procter & Gamble    Q212405     PG   pharmaceutical industry

In [ ]:
df_with_market_cap.to_csv("nyse_companies_with_market_cap_sorted.csv", index=False, encoding="utf-8")
print("Saved to nyse_nasdaq_companies_with_market_cap_sorted.csv")

Saved to nyse_companies_with_market_cap_sorted.csv


In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON

endpoint = "https://query.wikidata.org/sparql"

query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX p: <http://www.wikidata.org/prop/>
PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX bd: <http://www.bigdata.com/rdf#>

SELECT ?amount ?unitLabel ?date WHERE {
  wd:Q192314 p:P2226 ?statement .
  ?statement psv:P2226 ?value ;
             pq:P585 ?date .

  ?value wikibase:quantityAmount ?amount ;
         wikibase:quantityUnit ?unit .

  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY DESC(?date)
LIMIT 1
"""

sparql = SPARQLWrapper(endpoint, agent="MyApp/1.0 (your_email@example.com)")
sparql.setQuery(query)
sparql.setReturnFormat(JSON)

results = sparql.query().convert()
rows = results["results"]["bindings"]

if not rows:
    print("No market capitalization found in Wikidata.")
else:
    row = rows[0]
    amount = row["amount"]["value"]
    unit = row.get("unitLabel", {}).get("value", "")
    date = row["date"]["value"][:10]

    print(f"JPMorgan Chase market cap: {amount} {unit} (as of {date})")

JPMorgan Chase market cap: 686132000000 United States dollar (as of 2025-05-01)
